In [1]:
def get_action_on_state(op, num_qubits: int, context) -> list[complex]:
    """Returns state vector after applying given operation to zero state.

    Uses big-endian convention for numbering basis states.

    Args:
      op - Q# operation. Can be Q# callable from qdk.Context.code or evaluating other Q# callble.
        Also can be string, in which case it must evaluate to a callble.
        Signature of Q# callable must be (Qubit[] => Unit)
      num_qubits - number of qubits this operation acts on.

    Output: state vector, list of 2**num_qubits complex numbers.  
    """
    
    context.eval("""
       operation _GetActionOnZeroState(num_qubits: Int, op: (Qubit[] => Unit)) : Unit {
          use qubits = Qubit[num_qubits];
          op(qubits);
          Microsoft.Quantum.Diagnostics.DumpRegister(qubits);
          ResetAll(qubits);
       }
    """)    
    result = context.run(ctx.code._GetActionOnZeroState, 1, num_qubits, op, save_events=True)[0]
    state = result["events"][-1].state_dump().get_dict()
    result = [0.0]*(2**num_qubits)
    for key, value in state.items():
        result[key] = value
    return  result
    

In [2]:
# 1. Testing operation on single register, without any parameters.

import qdk

ctx = qdk.Context()
ctx.eval("""
operation MyOp1(q: Qubit[]) : Unit {
    H(q[0]);
    CNOT(q[0], q[1]);
    Z(q[1]);
}
""")

get_action_on_state(ctx.code.MyOp1, num_qubits=2, context=ctx)

[(0.7071067811865476+0j), 0.0, 0.0, (-0.7071067811865476-0j)]

In [3]:
# 2. Testing operation acting on 2 registers where we want to know their combined state.

import qdk

ctx = qdk.Context()
ctx.eval("""
operation MyOp2(q1: Qubit[], q2: Qubit[]) : Unit {
    H(q1[0]);
    CNOT(q1[0], q2[0]);
}

operation MyOp2_TestHelper(q: Qubit[]) : Unit {
    let n = Length(q);
    MyOp2(q[0..n/2-1], q[n/2..n-1]);
}
""")

get_action_on_state(ctx.code.MyOp2_TestHelper, num_qubits=4, context=ctx)

[(0.7071067811865476+0j),
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 (0.7071067811865476+0j),
 0.0,
 0.0,
 0.0,
 0.0,
 0.0]

In [4]:
# 3. Testing operation acting on 2 registers where we want to know state of just one register (partial trace).
# Note that this makes sense only when qubits in register we care about are not entangled with the rest, in which
# case it's valid to Reset all remaining qubits.
# See https://learn.microsoft.com/en-us/qsharp/api/qsharp-lang/std.diagnostics/dumpregister

import qdk

ctx = qdk.Context()
ctx.eval("""
operation MyOp3(q1: Qubit[], q2: Qubit[]) : Unit {
    H(q1[0]);
    H(q2[0]);
}

operation MyOp3_TestHelper(q: Qubit[]) : Unit {
    use q2 = Qubit[2];
    MyOp3(q, q2);
    ResetAll(q2);
}
""")

get_action_on_state(ctx.code.MyOp3_TestHelper, num_qubits=2, context=ctx)

[(0.7071067811865476+0j), 0.0, (0.7071067811865476+0j), 0.0]

In [5]:
# 4. Testing operation acting on non-zero initial state.
import qdk

ctx = qdk.Context()
ctx.eval("""
operation MyOp4(q: Qubit[])  : Unit is Adj {
    CNOT(q[0], q[1]);
    H(q[0]);
}

operation MyOp4_TestHelper(initial_state: Double[]) : (Qubit[] => Unit) {
    return q => {
      Std.StatePreparation.PreparePureStateD(initial_state, q);
      MyOp4(q);
    }
}
""")

s = 0.5**0.5
initial_state = [s,0,0,s]
get_action_on_state(ctx.code.MyOp4_TestHelper(initial_state), num_qubits=2, context=ctx)


[(1+0j), 0.0, 0.0, 0.0]

In [6]:
# 5. Testing operation taking parameters.
import qdk

ctx = qdk.Context()
ctx.eval("""
operation MyOp5(qs: Qubit[], angle: Double)  : Unit is Adj {
  for q in qs {
    Rx(angle, q);
  }
}

operation MyOp5_TestHelper(angle: Double) : (Qubit[] => Unit) {
    MyOp5(_, angle)
}
""")

get_action_on_state(ctx.code.MyOp5_TestHelper(0.3), num_qubits=2, context=ctx)

[(0.9776682445628031+0j),
 -0.1477601033306698j,
 -0.1477601033306698j,
 (-0.022331755437196992-0j)]

In [7]:
# This can be written without any helper:
get_action_on_state(ctx.eval("MyOp5(_, 0.3)"), num_qubits=2, context=ctx)

[(0.9776682445628031+0j),
 -0.1477601033306698j,
 -0.1477601033306698j,
 (-0.022331755437196992-0j)]